# Moteur de reconciliation - Mission 1

Chaine reproductible allant des deux sources brutes aux trois livrables.

**Regles de ce notebook**
- autonome : aucune variable heritee du notebook d'exploration
- lineaire : redemarrage du kernel puis execution complete doit rendre le meme resultat
- chaque section structurante se termine par son controle

**Perimetre vise** : lignes du fichier back office (9 010) + deals du front sans confirmation (150) = **9 160**

## Section 0 - Configuration

In [2]:
from pathlib import Path
import os

RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").is_dir())
os.chdir(RACINE)
print("racine projet :", RACINE)

racine projet : /Users/benjaminscemama/dev/market-risk-control


In [3]:
import sqlite3
import numpy as np
import pandas as pd

CHEMIN_DB = "data/risk.db"
CHEMIN_BO = "data/raw/bo_confirmations_20260724.csv"

DATE_REFERENCE = pd.Timestamp("2026-07-24")

## Section 1 - Lecture des sources

Le fichier back office s'oppose aux parametres par defaut de `read_csv` sur quatre points :
separateur `;`, decimale `,`, dates non parsees, et ordre jour/mois. Le quatrieme est celui qui
corrompt en silence.

Les colonnes de date du front arrivent de SQLite en **chaines de caracteres** et doivent etre
converties, sans quoi toute comparaison ou arithmetique de dates est fausse ou fragile.

In [4]:
con = sqlite3.connect(CHEMIN_DB)
fo = pd.read_sql("select * from trd_deal", con)
con.close()

for colonne in ["trade_date", "trade_ts", "delivery_start", "delivery_end"]:
    fo[colonne] = pd.to_datetime(fo[colonne])

len(fo), fo["deal_id"].nunique()

(9580, 9000)

In [5]:
bo = pd.read_csv(CHEMIN_BO, sep=";", decimal=",",
                 parse_dates=["trade_dt", "del_from", "del_to"], dayfirst=True)

len(bo), bo["deal_ref"].nunique()

(9010, 8945)

### Controle de section 1

Attendus : lignes du front (9 580) pour des identifiants distincts (9 000), lignes du back office
(9 010) pour des references distinctes (8 945), et types de date homogenes des deux cotes.

In [6]:
LIGNES_FO_ATTENDUES = 9580
DEALS_FO_ATTENDUS   = 9000
LIGNES_BO_ATTENDUES = 9010
REFS_BO_ATTENDUES   = 8945

DATES_FO = ["trade_date", "trade_ts", "delivery_start", "delivery_end"]
DATES_BO = ["trade_dt", "del_from", "del_to"]

assert len(fo) == LIGNES_FO_ATTENDUES, \
    f"lignes du front : {len(fo)} au lieu de {LIGNES_FO_ATTENDUES}"
assert fo["deal_id"].nunique() == DEALS_FO_ATTENDUS, \
    f"deal_id distincts : {fo['deal_id'].nunique()} au lieu de {DEALS_FO_ATTENDUS}"
assert len(bo) == LIGNES_BO_ATTENDUES, \
    f"lignes du back office : {len(bo)} au lieu de {LIGNES_BO_ATTENDUES}"
assert bo["deal_ref"].nunique() == REFS_BO_ATTENDUES, \
    f"deal_ref distinctes : {bo['deal_ref'].nunique()} au lieu de {REFS_BO_ATTENDUES}"

for colonne in DATES_FO:
    assert pd.api.types.is_datetime64_any_dtype(fo[colonne]), \
        f"front, {colonne} est en {fo[colonne].dtype} et non en datetime"
for colonne in DATES_BO:
    assert pd.api.types.is_datetime64_any_dtype(bo[colonne]), \
        f"back office, {colonne} est en {bo[colonne].dtype} et non en datetime"

assert fo.isna().sum().sum() == 0, f"front : {int(fo.isna().sum().sum())} valeurs manquantes"
assert bo.isna().sum().sum() == 0, f"back office : {int(bo.isna().sum().sum())} valeurs manquantes"

print("section 1 : les deux sources sont lues correctement")
print(f"  front       : {len(fo):5} lignes, {fo['deal_id'].nunique():5} deal_id distincts")
print(f"  back office : {len(bo):5} lignes, {bo['deal_ref'].nunique():5} deal_ref distinctes")

section 1 : les deux sources sont lues correctement
  front       :  9580 lignes,  9000 deal_id distincts
  back office :  9010 lignes,  8945 deal_ref distinctes


## Section 2 - Cle de reconciliation et selection de version

**Normalisation**, cote back office uniquement, le front etant propre sur ses 9 000 identifiants :
`strip`, puis `upper`, puis retrait du zero de remplissage `^D0+`.

**Selection de version** : derniere version par `deal_id`, puis filtre sur le statut. Interdit
d'arbitrer par `MAX(trade_ts)`, la colonne etant fabriquee sur les lignes amendees.

> Decision a ecrire : le front a joindre est-il l'ensemble des deals dans leur derniere version
> (9 000) ou seulement ceux dont la derniere version est confirmee (8 337) ? Justifier.

In [7]:
bo["deal_key"] = bo["deal_ref"].str.strip().str.upper()

non_conforme = ~bo["deal_key"].str.match(r"^D\d{7}$")
bo.loc[non_conforme, "deal_key"] = (
    bo.loc[non_conforme, "deal_key"].str.replace(r"^D0+", "D", regex=True)
)

bo["ref_corrigee"] = bo["deal_ref"] != bo["deal_key"]
bo

,confirmation_id,deal_ref,trade_dt,product,buy_sell,del_from,del_to,quantity,unit,unit_price,ccy,cpty_code,state,deal_key,ref_corrigee
0,BO905466,D2605829,2025-10-23,PWR_FR,B,2027-12-01,2028-11-30,255.5,MWH,91.995,EUR,TOTALE,MATCHED,D2605829,False
1,BO904587,D2604880,2026-02-19,PWR_FR,B,2026-07-01,2026-07-31,65.5,MWH,58.357,EUR,EEX_CL,MATCHED,D2604880,False
2,BO907311,D2607778,2026-02-10,NG_PEG,B,2027-12-01,2028-02-29,463.9,MWH,34.725,EUR,VITOL,MATCHED,D2607778,False
3,BO906063,D2606459,2026-03-10,NG_PEG,S,2026-05-01,2026-05-31,179.4,MWH,25.942,EUR,STATKR,MATCHED,D2606459,False
4,BO904238,D2604505,2025-12-30,PWR_FR,B,2027-08-01,2027-08-31,153.7,MWH,72.358,EUR,TOTALE,MATCHED,D2604505,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9005,BO903691,D2603923,2025-10-17,PWR_FR,S,2027-10-01,2027-10-31,1419.1,MWH,76.253,EUR,ICE_EN,MATCHED,D2603923,False
9006,BO908618,D2602952,2025-11-03,NG_PEG,S,2027-01-01,2027-01-31,106.4,MWH,38.337,EUR,MERCUR,MATCHED,D2602952,False
9007,BO906163,D2606564,2026-07-03,NG_PEG,B,2026-11-01,2026-11-30,31.0,MWH,30.690,EUR,ICE_EN,CXL,D2606564,False
9008,BO900414,D2600438,2026-05-07,NG_PEG,S,2027-09-01,2027-09-30,105.1,MWH,26.654,EUR,MERCUR,MATCHED,D2600438,False


In [8]:
fo_derniere_version = fo.sort_values(["deal_id", "version"]).drop_duplicates("deal_id", keep="last")
fo_derniere_version

,deal_id,trade_date,trade_ts,commodity,direction,delivery_start,delivery_end,volume_mwh,price_eur_mwh,counterparty,book,status,version
0,D2600000,2025-07-18,2025-07-18 13:44:28,POWER,BUY,2026-03-01,2026-03-31,605.2,92.459,STATKRAFT,B2B_FR_POWER_HEDGE,CONFIRMED,1
1,D2600001,2026-05-27,2026-05-27 15:03:14,GAS,SELL,2026-09-01,2026-09-30,61.1,22.523,VITOL,B2B_FR_GAS_HEDGE,CONFIRMED,1
2,D2600002,2025-07-25,2025-07-25 12:06:33,POWER,SELL,2026-02-01,2026-02-28,99.3,98.176,RWE,B2B_FR_POWER_HEDGE,CONFIRMED,1
3,D2600003,2025-08-19,2025-08-19 11:03:43,POWER,BUY,2026-08-01,2026-08-31,190.5,52.693,STATKRAFT,B2B_FR_POWER_HEDGE,CONFIRMED,1
4,D2600004,2025-08-07,2025-08-07 16:25:33,POWER,SELL,2026-11-01,2026-11-30,150.7,93.162,ICE_ENDEX,B2B_FR_POWER_HEDGE,PENDING,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8995,D2608995,2026-06-12,2026-06-12 13:20:36,GAS,BUY,2027-01-01,2027-03-31,525.2,37.383,UNIPER,B2B_FR_GAS_HEDGE,CONFIRMED,1
8996,D2608996,2025-06-04,2025-06-04 16:15:22,GAS,SELL,2027-01-01,2027-01-31,179.6,39.469,ICE_ENDEX,B2B_FR_GAS_HEDGE,CONFIRMED,1
8997,D2608997,2025-07-21,2025-07-21 12:28:53,GAS,SELL,2027-12-01,2027-12-31,1424.1,41.704,GUNVOR,B2B_FR_GAS_HEDGE,CONFIRMED,1
8998,D2608998,2025-09-30,2025-09-30 08:43:24,POWER,BUY,2026-05-01,2026-07-31,912.9,75.007,GUNVOR,B2B_FR_POWER_HEDGE,CONFIRMED,1


### Controle de section 2

Le cardinal doit etre conserve par la normalisation : references distinctes avant (8 945) et
apres (8 945). Une baisse signifierait que deux references distinctes ont ete fusionnees, donc
que la normalisation est trop laxiste.

In [9]:
refs_distinctes_avant = bo["deal_ref"].nunique()
refs_distinctes_apres = bo["deal_key"].nunique()

assert refs_distinctes_avant == refs_distinctes_apres, \
    f"la normalisation a fusionne des references distinctes : " \
    f"{refs_distinctes_apres} au lieu de {refs_distinctes_avant}"

## Section 3 - Table du perimetre

Jointure externe tracee, sur la cle normalisee du back office contre `deal_id`. La colonne de
provenance distingue les trois populations : appariees, confirmation sans deal, deal sans
confirmation.

La jointure se fait sur le front **deja dedoublonne par version**, sans quoi l'explosion de la
question 3 revient : une jointure naive rend 9 495 lignes pour 8 850 deals apparies.

In [10]:
perimetre = bo.merge(fo_derniere_version, left_on = "deal_key", right_on = "deal_id", how = "outer", indicator = True)
perimetre["_merge"].value_counts()

_merge
both          8915
right_only     150
left_only       95
Name: count, dtype: int64

### Controle de section 3

Le nombre de lignes doit valoir exactement le perimetre vise (9 160), verifiable par deux
chemins independants :

```
lignes du back office (9 010)  +  deals du front sans confirmation (150)
deals du front (9 000)  +  confirmations sans deal (95)  +  lignes dupliquees (65)
```

In [11]:
PERIMETRE_ATTENDU       = 9160
LIGNES_APPARIEES        = 8915
CONFIRMATIONS_SANS_DEAL = 95
DEALS_SANS_CONFIRMATION = 150
DEALS_APPARIES          = 8850

provenance = perimetre["_merge"].value_counts()
appariees               = provenance["both"]
confirmations_sans_deal = provenance["left_only"]
deals_sans_confirmation = provenance["right_only"]

deals_apparies    = perimetre.loc[perimetre["_merge"] == "both", "deal_id"].nunique()
lignes_dupliquees = appariees - deals_apparies

assert len(perimetre) == PERIMETRE_ATTENDU, \
    f"perimetre : {len(perimetre)} lignes au lieu de {PERIMETRE_ATTENDU}"
assert appariees == LIGNES_APPARIEES, \
    f"lignes appariees : {appariees} au lieu de {LIGNES_APPARIEES}"
assert confirmations_sans_deal == CONFIRMATIONS_SANS_DEAL, \
    f"confirmations sans deal : {confirmations_sans_deal} au lieu de {CONFIRMATIONS_SANS_DEAL}"
assert deals_sans_confirmation == DEALS_SANS_CONFIRMATION, \
    f"deals sans confirmation : {deals_sans_confirmation} au lieu de {DEALS_SANS_CONFIRMATION}"
assert deals_apparies == DEALS_APPARIES, \
    f"deals distincts apparies : {deals_apparies} au lieu de {DEALS_APPARIES}"

assert len(bo) + deals_sans_confirmation == PERIMETRE_ATTENDU, \
    (f"chemin back office : lignes du back office ({len(bo)}) "
     f"+ deals sans confirmation ({deals_sans_confirmation}) "
     f"= {len(bo) + deals_sans_confirmation}, attendu {PERIMETRE_ATTENDU}")
assert len(fo_derniere_version) + confirmations_sans_deal + lignes_dupliquees == PERIMETRE_ATTENDU, \
    (f"chemin front : deals du front ({len(fo_derniere_version)}) "
     f"+ confirmations sans deal ({confirmations_sans_deal}) "
     f"+ lignes dupliquees ({lignes_dupliquees}) "
     f"= {len(fo_derniere_version) + confirmations_sans_deal + lignes_dupliquees}, "
     f"attendu {PERIMETRE_ATTENDU}")

print("section 3 : le perimetre est complet et coherent par deux chemins")
print(f"  lignes appariees        : {appariees:5}  pour {deals_apparies} deals distincts")
print(f"  confirmations sans deal : {confirmations_sans_deal:5}")
print(f"  deals sans confirmation : {deals_sans_confirmation:5}")
print(f"  lignes dupliquees       : {lignes_dupliquees:5}")
print(f"  total                   : {len(perimetre):5}")

section 3 : le perimetre est complet et coherent par deux chemins
  lignes appariees        :  8915  pour 8850 deals distincts
  confirmations sans deal :    95
  deals sans confirmation :   150
  lignes dupliquees       :    65
  total                   :  9160


## Section 4 - Colonnes de controle

Une colonne booleenne par famille. Elles peuvent etre vraies simultanement a ce stade, c'est
normal : l'exclusivite n'intervient qu'a la section 5.

| Famille | Effectif mesure en exploration |
|---|---|
| Ligne dupliquee dans l'extrait | 65 |
| Confirmation sans deal | 95 |
| Deal sans confirmation | 150 |
| Quantite en kWh | 70 |
| Sens contradictoire | 55, dont 49 en vigueur |
| Ecart de prix materiel | 92 |
| Ecart de prix par arrondi | 106 |

Seuils retenus : ecart absolu de prix a **0,006 EUR/MWh**, un cran au-dessus de la borne
theorique de l'arrondi au centieme (0,005) que l'imprecision binaire franchit. Ecart relatif a
**0,1 %**.

In [12]:
perimetre["est_dupliquee"] = (
    perimetre["deal_key"].duplicated() & perimetre["deal_key"].notna()
)

perimetre["sans_deal"] = perimetre["_merge"] == "left_only"
perimetre["sans_confirmation"] = perimetre["_merge"] == "right_only"

rapport_quantite = perimetre["quantity"] / perimetre["volume_mwh"]
perimetre["unit_kwh"] = (
    np.isclose(rapport_quantite, 1000) & (perimetre["_merge"] == "both")
)

perimetre["sens_contradictoire"] = ((perimetre["buy_sell"].map({"B" : "BUY", "S":"SELL"}) != perimetre["direction"]) 
     & (perimetre["direction"].notna())
     & (perimetre["buy_sell"].notna()))


SEUIL_EUR = 0.006
apparie = perimetre["_merge"] == "both"
perimetre["ecart_prix_eur"] = perimetre["unit_price"] - perimetre["price_eur_mwh"]

perimetre["prix_arrondi"] = (
    apparie
    & (perimetre["ecart_prix_eur"] != 0)
    & np.isclose(perimetre["unit_price"], perimetre["price_eur_mwh"].round(2))
)

perimetre["prix_materiel"] = apparie & (perimetre["ecart_prix_eur"].abs() > SEUIL_EUR)

In [14]:
diff    = perimetre["unit_price"] != perimetre["price_eur_mwh"]
arrondi = np.isclose(perimetre["unit_price"], perimetre["price_eur_mwh"].round(2))

print("sans exiger un ecart :", (arrondi & apparie).sum())
print("en exigeant un ecart :", (arrondi & apparie & diff).sum())
print("lignes avec un ecart de prix quelconque :", (apparie & diff).sum())

sans exiger un ecart : 1041
en exigeant un ecart : 106
lignes avec un ecart de prix quelconque : 198


### Controle de section 4

Chaque effectif doit retomber sur celui mesure en exploration. Un ecart signale que la chaine
diverge, et le tableau ci-dessus dit entre quelles etapes chercher.

In [21]:
EFFECTIFS_ATTENDUS = {
    "est_dupliquee": 65,
    "sans_deal": 95,
    "sans_confirmation" : 150,
    "unit_kwh": 70,
    "sens_contradictoire": 55,
    "prix_materiel": 92,
    "prix_arrondi" : 106,
}
COLONNES_CONTROLE = list(EFFECTIFS_ATTENDUS)
ECARTS_DE_PRIX_ATTENDUS = 198

for colonne, attendu in EFFECTIFS_ATTENDUS.items():
    obtenu = int(perimetre[colonne].sum())
    assert obtenu == attendu, f"{colonne} : {obtenu} lignes au lieu de {attendu}"

manquantes = perimetre[COLONNES_CONTROLE].isna().sum()
assert manquantes.sum() == 0, \
    f"valeurs manquantes dans les colonnes de controle :\n{manquantes[manquantes > 0]}"

for colonne in COLONNES_CONTROLE:
    assert perimetre[colonne].dtype == bool, \
        f"{colonne} est en {perimetre[colonne].dtype} et non en booleen"

ecarts_de_prix = int((apparie & (perimetre["ecart_prix_eur"] != 0)).sum())
assert perimetre["prix_arrondi"].sum() + perimetre["prix_materiel"].sum() == ecarts_de_prix, \
    (f"arrondis ({int(perimetre['prix_arrondi'].sum())}) "
     f"+ materiels ({int(perimetre['prix_materiel'].sum())}) "
     f"ne font pas les ecarts de prix ({ecarts_de_prix})")
assert ecarts_de_prix == ECARTS_DE_PRIX_ATTENDUS, \
    f"ecarts de prix : {ecarts_de_prix} au lieu de {ECARTS_DE_PRIX_ATTENDUS}"
assert (perimetre["prix_arrondi"] & perimetre["prix_materiel"]).sum() == 0, \
    "une ligne est a la fois expliquee par un arrondi et materielle : le seuil est mal place"

assert (perimetre["unit_kwh"] == (perimetre["unit"] == "KWH")).all(), \
    "la detection d'unite par le rapport ne coincide plus avec la colonne unit"

print("section 4 : les sept colonnes de controle sont conformes")
for colonne, attendu in EFFECTIFS_ATTENDUS.items():
    print(f"  {colonne:22} {attendu:5}")

section 4 : les sept colonnes de controle sont conformes
  est_dupliquee             65
  sans_deal                 95
  sans_confirmation        150
  unit_kwh                  70
  sens_contradictoire       55
  prix_materiel             92
  prix_arrondi             106


## Section 5 - Categorie unique

**Regle B, par ordre de correction.** Chaque rang est une condition prealable au suivant : tant
que le defaut de rang n n'est pas corrige, l'analyse du rang n+1 n'a pas de sens.

| Rang | Categorie | Pourquoi a ce rang |
|---|---|---|
| 1 | Ligne dupliquee dans l'extrait | dedoublonner avant toute mesure, sinon tout compte double |
| 2 | Confirmation sans deal, deal sans confirmation | sans rapprochement, aucune comparaison n'existe |
| 3 | Quantite en kWh | unite fausse, donc volume et impact faux d'un facteur 1 000 |
| 4 | Sens contradictoire | affecte la position, donc l'agregat |
| 5 | Ecart de prix materiel | n'affecte ni volume ni position, seulement la valorisation |
| 6 | Ecart de prix par arrondi | n'est pas un ecart |
| 7 | Concordante | aucun defaut |

**Alternative ecartee : la regle A, par gravite decroissante.** Elle classerait chaque ligne dans
la famille de plus fort impact. Ecartee parce qu'elle compare des grandeurs heterogenes, des MWh
pour une unite fausse et des euros pour un ecart de prix, et parce que le classement d'une ligne
changerait avec les prix, rendant la synthese non reproductible.

In [37]:
LIBELLES = {
    "est_dupliquee":       "ligne dupliquee dans l'extrait",
    "sans_deal":           "confirmation sans deal",
    "sans_confirmation":   "deal sans confirmation",
    "unit_kwh":            "quantite en kWh",
    "sens_contradictoire": "sens contradictoire",
    "prix_materiel":       "ecart de prix materiel",
    "prix_arrondi":        "ecart de prix par arrondi",
}

perimetre["categorie"] = np.select(
    [perimetre[colonne] for colonne in LIBELLES],
    list(LIBELLES.values()),
    default="concordante",
)

In [46]:
print(perimetre["categorie"].value_counts())
print()
print(perimetre["categorie"].value_counts().sum())

categorie
concordante                       8535
deal sans confirmation             150
ecart de prix par arrondi          102
confirmation sans deal              95
ecart de prix materiel              89
quantite en kWh                     70
ligne dupliquee dans l'extrait      65
sens contradictoire                 54
Name: count, dtype: int64

9160


In [39]:
nb_defauts = perimetre[list(LIBELLES)].sum(axis=1)
nb_defauts.value_counts().sort_index().rename("lignes")

0    8535
1     618
2       6
3       1
Name: lignes, dtype: int64

In [40]:
classe = perimetre["categorie"].value_counts()

absorption = pd.DataFrame({
    "brut":   {lib: int(perimetre[col].sum()) for col, lib in LIBELLES.items()},
    "classe": {lib: int(classe.get(lib, 0))   for lib in LIBELLES.values()},
})
absorption["absorbees"] = absorption["brut"] - absorption["classe"]
absorption

,brut,classe,absorbees
ligne dupliquee dans l'extrait,65,65,0
confirmation sans deal,95,95,0
deal sans confirmation,150,150,0
quantite en kWh,70,70,0
sens contradictoire,55,54,1
ecart de prix materiel,92,89,3
ecart de prix par arrondi,106,102,4


### Controle de section 5

Aucune ligne sans categorie, et la somme des effectifs par categorie doit valoir exactement le
perimetre (9 160). C'est le controle d'exhaustivite et d'exclusivite reuni.

In [55]:
effectifs  = perimetre["categorie"].value_counts()
nb_defauts = perimetre[list(LIBELLES)].sum(axis=1)

assert effectifs.sum() == PERIMETRE_ATTENDU, \
    f"somme des categories : {effectifs.sum()} au lieu de {PERIMETRE_ATTENDU}"

assert perimetre["categorie"].isna().sum() == 0, \
    f"{perimetre['categorie'].isna().sum()} lignes sans categorie"
assert (perimetre["categorie"] == "").sum() == 0, \
    f"{(perimetre['categorie'] == '').sum()} lignes a categorie vide"

for colonne, libelle in LIBELLES.items():
    brut, classe = int(perimetre[colonne].sum()), int(effectifs.get(libelle, 0))
    assert classe <= brut, \
        f"{libelle} : {classe} lignes classees pour {brut} detectees, la cascade est mal ordonnee"

concordantes_attendues = int((nb_defauts == 0).sum())
assert effectifs["concordante"] == concordantes_attendues, \
    (f"concordantes : {effectifs['concordante']} classees "
     f"pour {concordantes_attendues} lignes sans aucun defaut")

absorbees        = sum(int(perimetre[c].sum()) - int(effectifs.get(l, 0)) for c, l in LIBELLES.items())
defauts_en_exces = int((nb_defauts - 1).clip(lower=0).sum())
assert absorbees == defauts_en_exces, \
    (f"absorbees par la cascade ({absorbees}) "
     f"different des defauts en exces ({defauts_en_exces})")

print("section 5 : categories exclusives et exhaustives")
print(f"  lignes classees      : {effectifs.sum()}")
print(f"  lignes sans defaut   : {concordantes_attendues}")
print(f"  defauts en exces     : {defauts_en_exces}, absorbes par la cascade : {absorbees}")

section 5 : categories exclusives et exhaustives
  lignes classees      : 9160
  lignes sans defaut   : 8535
  defauts en exces     : 8, absorbes par la cascade : 8


## Section 6 - Impact

> Decision a ecrire : convention de calcul de l'impact, uniforme et applicable a toutes les
> categories, en MWh et en euros. Sans convention unique, la colonne d'impact de la synthese
> additionne des grandeurs differentes.

Rappel des ordres de grandeur mesures en exploration : erreur d'unite si on ne convertit pas
(20 931 448 MWh), largeur d'indetermination sur les sens (85,6 MWh), impact brut des ecarts de
prix materiels en vigueur (47 949,91 EUR) pour un impact net de 5 576,93 EUR.

In [121]:
perimetre.columns

Index(['confirmation_id', 'deal_ref', 'trade_dt', 'product', 'buy_sell',
       'del_from', 'del_to', 'quantity', 'unit', 'unit_price', 'ccy',
       'cpty_code', 'state', 'deal_key', 'ref_corrigee', 'deal_id',
       'trade_date', 'trade_ts', 'commodity', 'direction', 'delivery_start',
       'delivery_end', 'volume_mwh', 'price_eur_mwh', 'counterparty', 'book',
       'status', 'version', '_merge', 'est_dupliquee', 'sans_deal',
       'sans_confirmation', 'unit_kwh', 'sens_contradictoire',
       'ecart_prix_eur', 'prix_arrondi', 'prix_materiel', 'categorie',
       'notionnel_eur', 'notionnel_front', 'notionnel_bo', 'volume_kwh_exces',
       'impact_mwh', 'impact_eur', 'indetermination_mwh'],
      dtype='object')

In [122]:
EN_VIGUEUR = pd.Series(
    np.where(perimetre["sans_deal"],
             perimetre["state"]  == "MATCHED",
             perimetre["status"] == "CONFIRMED"),
    index=perimetre.index,
)

perimetre["notionnel_front"]  = perimetre["volume_mwh"] * perimetre["price_eur_mwh"]
perimetre["notionnel_bo"]     = perimetre["quantity"]   * perimetre["unit_price"]
perimetre["volume_kwh_exces"] = perimetre["quantity"]   - perimetre["volume_mwh"]

cat = perimetre["categorie"]

perimetre["impact_mwh"] = np.select(
    [cat == "ligne dupliquee dans l'extrait",
     cat == "confirmation sans deal",
     cat == "deal sans confirmation",
     cat == "quantite en kWh"],
    [perimetre["volume_mwh"],
     perimetre["quantity"],
     perimetre["volume_mwh"],
     perimetre["volume_kwh_exces"]],
    default=0.0)

perimetre["impact_eur"] = np.select(
    [cat == "ligne dupliquee dans l'extrait",
     cat == "confirmation sans deal",
     cat == "deal sans confirmation",
     cat == "quantite en kWh",
     cat == "ecart de prix materiel",
     cat == "ecart de prix par arrondi"],
    [perimetre["notionnel_front"],
     perimetre["notionnel_bo"],
     perimetre["notionnel_front"],
     perimetre["volume_kwh_exces"] * perimetre["price_eur_mwh"],
     perimetre["ecart_prix_eur"]   * perimetre["volume_mwh"],
     perimetre["ecart_prix_eur"]   * perimetre["volume_mwh"]],
    default=0.0)

signe = np.where(perimetre["direction"] == "BUY", 1.0, -1.0)
perimetre["indetermination_mwh"] = np.where(
    (cat == "sens contradictoire") & EN_VIGUEUR,
    2 * signe * perimetre["volume_mwh"], 0.0)

for colonne in ["impact_mwh", "impact_eur", "indetermination_mwh"]:
    perimetre[colonne] = perimetre[colonne].fillna(0.0)
    perimetre.loc[~EN_VIGUEUR, colonne] = 0.0

## Section 7 - Les trois livrables

1. **Table de reconciliation ligne a ligne** : une ligne par element du perimetre (9 160), avec
   sa categorie et son impact.
2. **Synthese par categorie** : effectif, impact en MWh, impact en euros, avec ligne de total.
3. **Liste d'anomalies** : triee par impact absolu decroissant, avec part cumulee, destinee au
   back office.

In [142]:
perimetre.columns

Index(['confirmation_id', 'deal_ref', 'trade_dt', 'product', 'buy_sell',
       'del_from', 'del_to', 'quantity', 'unit', 'unit_price', 'ccy',
       'cpty_code', 'state', 'deal_key', 'ref_corrigee', 'deal_id',
       'trade_date', 'trade_ts', 'commodity', 'direction', 'delivery_start',
       'delivery_end', 'volume_mwh', 'price_eur_mwh', 'counterparty', 'book',
       'status', 'version', '_merge', 'est_dupliquee', 'sans_deal',
       'sans_confirmation', 'unit_kwh', 'sens_contradictoire',
       'ecart_prix_eur', 'prix_arrondi', 'prix_materiel', 'categorie',
       'notionnel_eur', 'notionnel_front', 'notionnel_bo', 'volume_kwh_exces',
       'impact_mwh', 'impact_eur', 'indetermination_mwh'],
      dtype='object')

In [144]:
perimetre[["categorie", "impact_mwh", "impact_eur", "indetermination_mwh"]]

,categorie,impact_mwh,impact_eur,indetermination_mwh
0,concordante,0.0,0.0000,0.0
1,concordante,0.0,0.0000,0.0
2,concordante,0.0,0.0000,0.0
3,concordante,0.0,0.0000,0.0
4,concordante,0.0,0.0000,0.0
...,...,...,...,...
9155,confirmation sans deal,507.7,35333.8892,0.0
9156,confirmation sans deal,164.4,12156.5580,0.0
9157,confirmation sans deal,0.0,0.0000,0.0
9158,confirmation sans deal,164.1,15474.9582,0.0


In [150]:
synthese = perimetre.groupby("categorie").agg(
    lignes=("categorie", "size"),
    en_vigueur=("categorie", lambda s: int(EN_VIGUEUR.loc[s.index].sum())),
    impact_mwh_net=("impact_mwh", "sum"),
    impact_mwh_brut=("impact_mwh", lambda s: s.abs().sum()),
    impact_eur_net=("impact_eur", "sum"),
    impact_eur_brut=("impact_eur", lambda s: s.abs().sum()),
)

synthese = synthese.sort_values("impact_eur_brut", ascending=False)
synthese.loc["TOTAL"] = synthese.sum()
synthese[["lignes", "en_vigueur"]] = synthese[["lignes", "en_vigueur"]].astype(int)

assert synthese.loc["TOTAL", "lignes"] == PERIMETRE_ATTENDU, \
    f"synthese : {synthese.loc['TOTAL', 'lignes']} lignes au lieu de {PERIMETRE_ATTENDU}"

synthese

,lignes,en_vigueur,impact_mwh_net,impact_mwh_brut,impact_eur_net,impact_eur_brut
categorie,,,,,,
quantite en kWh,70,61,15863121.0,15863121.0,9.941873e+08,9.941873e+08
deal sans confirmation,150,133,35636.5,35636.5,2.298106e+06,2.298106e+06
confirmation sans deal,95,91,34098.8,34098.8,1.751654e+06,1.751654e+06
ligne dupliquee dans l'extrait,65,61,20192.9,20192.9,1.146076e+06,1.146076e+06
ecart de prix materiel,89,81,0.0,0.0,-9.376042e+03,4.361649e+04
ecart de prix par arrondi,102,91,0.0,0.0,-3.019650e+01,8.642810e+01
concordante,8535,7922,0.0,0.0,0.000000e+00,0.000000e+00
sens contradictoire,54,49,0.0,0.0,0.000000e+00,0.000000e+00
TOTAL,9160,8489,15953049.2,15953049.2,9.993738e+08,9.994269e+08


In [154]:
perimetre["cle"] = perimetre["deal_key"].fillna(perimetre["deal_id"])
assert perimetre["cle"].notna().all(), "des lignes sans cle unifiee"

HORS_LISTE = ["concordante", "ecart de prix par arrondi"]

anomalies = perimetre.loc[
    EN_VIGUEUR & ~perimetre["categorie"].isin(HORS_LISTE),
    ["cle", "deal_ref", "confirmation_id", "categorie",
     "quantity", "volume_mwh", "unit_price", "price_eur_mwh",
     "ecart_prix_eur", "impact_mwh", "impact_eur"],
].copy()

anomalies = anomalies.reindex(
    anomalies["impact_eur"].abs().sort_values(ascending=False).index)

anomalies.insert(0, "rang", range(1, len(anomalies) + 1))
anomalies["part_cumulee"] = (anomalies["impact_eur"].abs().cumsum()
                             / anomalies["impact_eur"].abs().sum())

anomalies

,rang,cle,deal_ref,confirmation_id,categorie,quantity,volume_mwh,unit_price,price_eur_mwh,ecart_prix_eur,impact_mwh,impact_eur,part_cumulee
1152,1,D2601148,D2601148,BO901078,quantite en kWh,827700.0,827.7,86.984,86.984,0.0,826872.3,7.192466e+07,0.071966
5202,2,D2605166,D2605166,BO904846,quantite en kWh,679600.0,679.6,98.193,98.193,0.0,678920.4,6.666523e+07,0.138669
6102,3,D2606059,D2606059,BO905680,quantite en kWh,898200.0,898.2,73.673,73.673,0.0,897301.8,6.610692e+07,0.204814
5710,4,D2605670,D2605670,BO905317,quantite en kWh,498900.0,498.9,93.491,93.491,0.0,498401.1,4.659602e+07,0.251437
3960,5,D2603932,D2603932,BO903700,quantite en kWh,452800.0,452.8,102.073,102.073,0.0,452347.2,4.617244e+07,0.297636
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4683,472,D2604653,D2604653,BO904376,sens contradictoire,191.3,191.3,85.326,85.326,0.0,0.0,0.000000e+00,1.000000
4766,473,D2604735,D2604735,BO904451,sens contradictoire,159.5,159.5,105.238,105.238,0.0,0.0,0.000000e+00,1.000000
4957,474,D2604922,D2604922,BO904627,sens contradictoire,136.5,136.5,35.225,35.225,0.0,0.0,0.000000e+00,1.000000
5124,475,D2605088,D2605088,BO904774,sens contradictoire,205.0,205.0,34.536,34.536,0.0,0.0,0.000000e+00,1.000000
